In [ ]:
import os
import csv
import requests
import time
import random


GITHUB_TOKEN = ""

TOTAL_REPOS_TO_FIND = 200
CSV_FILE = '../results/data/individual_repositories.csv'
API_URL = 'https://api.github.com/search/repositories'

# Target languages and their distribution
TARGET_LANGUAGES = {
    'C': 400,
    'C++': 400,
    'C#': 400,
    'Java': 400,
    'JavaScript': 400,
    'Python': 400
}

# Language counters
language_counts = {lang: 0 for lang in TARGET_LANGUAGES.keys()}

# Blacklist for known organization/foundation account names BLACKLISTED_OWNERS
EXCLUDED_OWNERS = {
    'google', 'microsoft', 'facebook', 'apple', 'amazon', 'netflix', 'ibm',
    'oracle', 'intel', 'adobe', 'airbnb', 'uber', 'linkedin', 'twitter',
    'mozilla', 'apache', 'torvalds', 'docker', 'kubernetes', 'tensorflow',
    'pytorch', 'angular', 'vuejs', 'reactjs', 'nodejs', 'golang', 'rust-lang',
    'jetbrains', 'elastic', 'mongodb', 'automattic', 'square', 'shopify',
    'stripe', 'spotify', 'dropbox', 'github', 'gitlab', 'atlassian', 'slack',
    'zoom', 'salesforce', 'unity', 'unreal', 'epic', 'valve', 'steam',
    # Additional major tech companies and organizations
    'alphabet', 'samsung', 'samsungelectronics', 'honhai', 'honhaiprecision',
    'meta', 'metaplatforms', 'huawei', 'huaweiinvestment', 'sony', 'dell',
    'delltechnologies', 'tencent', 'tencentholdings', 'taiwan', 'taiwansemiconductor',
    'tsmc', 'hitachi', 'lg', 'lgelectronics', 'accenture', 'nvidia', 'panasonic',
    'panasonicholdings', 'cisco', 'ciscosystems', 'lenovo', 'lenovogroup',
    'hp', 'pegatron', 'xiaomi', 'ubertech', 'ubertechnologies', 'qualcomm',
    'broadcom', 'chinaelectronics', 'quanta', 'quantacomputer', 'jabil',
    'sap', 'sapse', 'luxshare', 'luxshareprecision',
    # Open source foundations and organizations
    'mozilla', 'apache', 'gnu', 'gnulinux', 'linuxfoundation', 'oniro',
    'railcasts', 'cloudnative', 'cncf', 'gnome', 'kde', 'openstack',
    'osgeo', 'opensourcegeospatial', 'softwareheritage', 'openknowledge',
    'wikimedia', 'ourresearch', 'berkeley', 'mit', 'stanford', 'stanforduniversity',
    'audiopedia', 'audiopediafoundation', 'opensourcesecurity', 'openssf',
    'openjs', 'openjsfoundation', 'academysoftware', 'academysoftwarefoundation',
    'openmobility', 'openmobilityfoundation', 'osu', 'opensource', 'fossi',
    'fossifoundation', 'openwallet', 'openwalletfoundation', 'verapdf',
    'nomic', 'nomicfoundation', 'farama', 'faramafoundation', 'fintech',
    'fintechopensource', 'communityox', 'commonhaus'
}

def check_rate_limit():
    """Check remaining rate limit."""
    headers = {'Authorization': f'token {GITHUB_TOKEN}'} if GITHUB_TOKEN else {}
    try:
        response = requests.get('https://api.github.com/rate_limit', headers=headers)
        if response.status_code == 200:
            data = response.json()
            search_remaining = data['resources']['search']['remaining']
            search_reset = data['resources']['search']['reset']
            print(f"Search API remaining: {search_remaining}")
            return search_remaining, search_reset
        return 0, 0
    except:
        return 0, 0

def wait_for_rate_limit_reset(reset_time):
    """Wait until rate limit resets."""
    current_time = time.time()
    wait_time = max(0, reset_time - current_time + 10)  # Add 10 seconds buffer
    if wait_time > 0:
        print(f"Rate limit exceeded. Waiting {wait_time:.0f} seconds...")
        time.sleep(wait_time)

def search_github_repos(query, page):
    """Sends a search request to the GitHub API with better error handling."""
    headers = {
        'Accept': 'application/vnd.github.v3+json',
        'User-Agent': 'GitHub-Repo-Scraper'
    }
    if GITHUB_TOKEN:
        headers['Authorization'] = f'token {GITHUB_TOKEN}'
    
    params = {
        'q': query,
        'sort': 'stars',
        'order': 'desc',
        'per_page': 100,
        'page': page
    }
    
    try:
        response = requests.get(API_URL, headers=headers, params=params)
        
        if response.status_code == 403:
            remaining, reset_time = check_rate_limit()
            if remaining == 0:
                wait_for_rate_limit_reset(reset_time)
                # Retry the request
                response = requests.get(API_URL, headers=headers, params=params)
            else:
                print(f"403 error but rate limit shows {remaining} remaining. Waiting 60 seconds...")
                time.sleep(60)
                return None
        
        response.raise_for_status()
        return response.json()
        
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None

def generate_language_queries():
    """Generate queries specifically for target languages with floor-only limits."""
    queries = []
    
    # GitHub API language names mapping
    language_mapping = {
        'C': 'c',
        'C++': 'cpp',
        'C#': 'csharp',
        'Java': 'java',
        'JavaScript': 'javascript',
        'Python': 'python'
    }
    
    # Different query patterns with floor-only limits
    query_patterns = [
        'language:{} type:user stars:>=5',           # At least 5 stars
        'language:{} type:user stars:>=10',          # At least 10 stars  
        'language:{} type:user stars:>=3 forks:>=3', # At least 3 stars, 1 fork
        'language:{} type:user stars:>=5 size:1024..10000', # Keep size limit for practical reasons
        #'language:{} type:user created:2020-01-01..2024-12-31 stars:>=5',
        #'language:{} type:user created:2021-01-01..2024-12-31 stars:>=3',
        'language:{} type:user forks:>=1 stars:>=5', # At least 1 fork, 5 stars
        'language:{} type:user forks:>=2 stars:>=10', # At least 2 forks, 10 stars
        'language:{} type:user stars:>=1 forks:>=1', # Basic quality floor
        'language:{} type:user stars:>=20',          # Higher quality repos
        'language:{} type:user stars:>=50',          # Even higher quality
    ]
    
    # Generate queries for each target language
    for lang_display, lang_api in language_mapping.items():
        for pattern in query_patterns:
            queries.append((pattern.format(lang_api), lang_display))
    
    # Topic-based queries with floor limits
    topics = ['tutorial', 'learning', 'practice', 'project', 'example', 'beginner', 'advanced']
    for lang_display, lang_api in language_mapping.items():
        for topic in topics:
            queries.append((f'topic:{topic} language:{lang_api} type:user stars:>=3', lang_display))
    
    # Add some queries targeting different activity levels
    activity_queries = [
        'language:{} type:user pushed:>=2023-01-01 stars:>=5',  # Recently active
        'language:{} type:user issues:>=1 stars:>=10',          # Has community engagement
        'language:{} type:user watchers:>=2 stars:>=5',         # Being watched
    ]
    
    for lang_display, lang_api in language_mapping.items():
        for pattern in activity_queries:
            queries.append((pattern.format(lang_api), lang_display))
    
    # Shuffle to add randomness
    random.shuffle(queries)
    return queries

def is_target_complete():
    """Check if we've reached the target for all languages."""
    return all(count >= target for count, target in zip(language_counts.values(), TARGET_LANGUAGES.values()))

def get_priority_languages():
    """Get languages that still need more repositories, prioritizing those furthest from target."""
    needed = []
    for lang, target in TARGET_LANGUAGES.items():
        current = language_counts[lang]
        if current < target:
            needed.append((lang, target - current))
    
    # Sort by how many more we need (descending)
    needed.sort(key=lambda x: x[1], reverse=True)
    return [lang for lang, _ in needed]

def main():
    """Main function to find repos and write to CSV."""
    if not GITHUB_TOKEN:
        print("Error: GITHUB_TOKEN environment variable not set.")
        return

    print(f"Starting repository search. Goal: {TOTAL_REPOS_TO_FIND} repositories.")
    print("Target distribution by language:")
    for lang, target in TARGET_LANGUAGES.items():
        print(f"  {lang}: {target} repositories")
    
    # Check initial rate limit
    remaining, reset_time = check_rate_limit()
    print(f"Starting with {remaining} API calls remaining")

    repo_urls = set()
    found_repos = []
    file_exists = os.path.isfile(CSV_FILE)

    # Load existing repos and count by language
    if file_exists:
        print(f"Reading existing repositories from {CSV_FILE}...")
        with open(CSV_FILE, 'r', newline='', encoding='utf-8') as csvfile:
            try:
                csv_reader = csv.reader(csvfile)
                header = next(csv_reader)
                for row in csv_reader:
                    if len(row) > 3:
                        repo_urls.add(row[2])
                        found_repos.append(row)
                        # Count existing repositories by language
                        repo_language = row[3]
                        if repo_language in TARGET_LANGUAGES:
                            language_counts[repo_language] += 1
            except StopIteration:
                pass
        print(f"Found {len(found_repos)} existing repositories.")
        print("Current language distribution:")
        for lang, count in language_counts.items():
            print(f"  {lang}: {count}/{TARGET_LANGUAGES[lang]}")

    if is_target_complete():
        print("All language targets have been reached!")
        return

    queries = generate_language_queries()
    print(f"Generated {len(queries)} language-specific queries")

    with open(CSV_FILE, 'a', newline='', encoding='utf-8') as csvfile:
        csv_writer = csv.writer(csvfile)
        
        if not file_exists or os.path.getsize(CSV_FILE) == 0:
            csv_writer.writerow(['Owner account Name', 'Repo Name', 'Link', 'Language of repo'])

        consecutive_failures = 0
        
        for query_idx, (query, target_lang) in enumerate(queries):
            if is_target_complete():
                print("All language targets reached!")
                break
            
            # Skip queries for languages that have reached their target
            if language_counts[target_lang] >= TARGET_LANGUAGES[target_lang]:
                continue
            
            print(f"\nQuery {query_idx + 1}/{len(queries)}: '{query}' (Target: {target_lang})")
            print(f"Current progress - {target_lang}: {language_counts[target_lang]}/{TARGET_LANGUAGES[target_lang]}")
            print(f"Total repositories: {sum(language_counts.values())}/{TOTAL_REPOS_TO_FIND}")
            
            # Check rate limit before starting new query
            remaining, reset_time = check_rate_limit()
            if remaining < 5:  # Leave some buffer
                print("Low on API calls, waiting for reset...")
                wait_for_rate_limit_reset(reset_time)
            
            query_found_new = False
            
            for page in range(1, 6):  # Check up to 5 pages per query
                if language_counts[target_lang] >= TARGET_LANGUAGES[target_lang]:
                    break

                print(f"  Page {page}...")
                data = search_github_repos(query, page)

                if not data:
                    consecutive_failures += 1
                    print(f"  Failed to get data (consecutive failures: {consecutive_failures})")
                    if consecutive_failures >= 5:
                        print("Too many consecutive failures, taking a longer break...")
                        time.sleep(300)  # 5 minute break
                        consecutive_failures = 0
                    break

                consecutive_failures = 0

                if 'items' not in data or not data['items']:
                    print(f"  No items on page {page}")
                    break

                new_repos_this_page = 0
                for item in data['items']:
                    if language_counts[target_lang] >= TARGET_LANGUAGES[target_lang]:
                        break
                        
                    owner_name = item['owner']['login']
                    owner_type = item['owner']['type']
                    repo_language = item['language']
                    
                    # More strict filtering - only accept repos with target languages
                    if (item['html_url'] not in repo_urls and 
                        owner_name.lower() not in EXCLUDED_OWNERS and
                        owner_type == 'User' and  # Only individual users
                        repo_language in TARGET_LANGUAGES and
                        language_counts[repo_language] < TARGET_LANGUAGES[repo_language]):
                        
                        repo_name = item['name']
                        link = item['html_url']
                        
                        repo_data = [owner_name, repo_name, link, repo_language]
                        found_repos.append(repo_data)
                        repo_urls.add(link)
                        csv_writer.writerow(repo_data)
                        csvfile.flush()  # Ensure data is written immediately
                        
                        # Update language count
                        language_counts[repo_language] += 1
                        new_repos_this_page += 1
                        query_found_new = True

                        if sum(language_counts.values()) % 25 == 0:
                            print(f"    Progress: {sum(language_counts.values())}/{TOTAL_REPOS_TO_FIND} repositories")
                            print(f"    Language distribution: {dict(language_counts)}")

                print(f"  Added {new_repos_this_page} new repos from page {page}")
                
                if new_repos_this_page == 0:
                    break  # No point checking more pages if no new repos

                # Respectful delay
                time.sleep(random.uniform(3, 6))

            if not query_found_new:
                print(f"  No new repositories found for this query")
            
            # Longer delay between queries
            time.sleep(random.uniform(5, 10))

    print(f"\nFinished! Total repositories found: {sum(language_counts.values())}")
    print("Final language distribution:")
    for lang, count in language_counts.items():
        target = TARGET_LANGUAGES[lang]
        percentage = (count / target) * 100 if target > 0 else 0
        print(f"  {lang}: {count}/{target} ({percentage:.1f}%)")
    print(f"Data saved to {CSV_FILE}")
    
    # Final rate limit check
    remaining, _ = check_rate_limit()
    print(f"Remaining API calls: {remaining}")

if __name__ == '__main__':
    main()

Starting repository search. Goal: 200 repositories.
Target distribution by language:
  C: 400 repositories
  C++: 400 repositories
  C#: 400 repositories
  Java: 400 repositories
  JavaScript: 400 repositories
  Python: 400 repositories
Search API remaining: 30
Starting with 30 API calls remaining
Generated 126 language-specific queries

Query 1/126: 'language:java type:user watchers:>=2 stars:>=5' (Target: Java)
Current progress - Java: 0/400
Total repositories: 0/200
Search API remaining: 30
  Page 1...
  No items on page 1
  No new repositories found for this query

Query 2/126: 'language:java type:user forks:>=1 stars:>=5' (Target: Java)
Current progress - Java: 0/400
Total repositories: 0/200
Search API remaining: 29
  Page 1...
    Progress: 25/200 repositories
    Language distribution: {'C': 0, 'C++': 0, 'C#': 0, 'Java': 25, 'JavaScript': 0, 'Python': 0}
  Added 43 new repos from page 1
  Page 2...
  No items on page 2

Query 3/126: 'language:java type:user stars:>=1 forks:>=1'